# 01 — Extração de características (Kaggle, CPU)

Gera `features/slices/*.parquet` (~3.000 exames, 1 a cada 2 cortes ≈ 370 mil cortes).
Tempo estimado: 2–3 h em CPU de 4 núcleos. **Retomável**: se a sessão cair, rode de novo.

Para paralelizar em várias sessões, use `N_SHARDS > 1` e um `SHARD` diferente em cada cópia
deste notebook; depois junte as saídas como Inputs do notebook 02.

Ao final: **Save Version → Save & Run All**; a saída (`/kaggle/working/ap2-topicos/features`)
vira um Dataset que o notebook 02 usa como Input.

In [ ]:
# === Setup (Kaggle) ==========================================================
# Pré-requisitos: aceitar as regras da competição "RSNA STR Pulmonary Embolism Detection"
# e adicioná-la como Input deste notebook (+ Add Input -> Competitions).
import os, sys, subprocess, shutil
REPO_URL = "https://github.com/silvarawenne/ap2-topicos.git"
REPO_BRANCH = "main"                      # troque se estiver usando outra branch
REPO = "/kaggle/working/ap2-topicos"
CODE_DATASET = "/kaggle/input/ap2-topicos-code"   # alternativa sem Internet: repo enviado como Dataset

if not os.path.exists(REPO):
    if os.path.exists(CODE_DATASET):
        shutil.copytree(CODE_DATASET, REPO)
    else:  # exige "Internet on" nas configurações do notebook
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-kaggle.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
print("repo:", REPO, "| dados:", os.listdir("/kaggle/input"))

In [ ]:
SHARD, N_SHARDS = 0, 1
!python scripts/01_make_split_and_sample.py
!python scripts/02_extract_features.py --shard {SHARD} --n-shards {N_SHARDS}

In [ ]:
import pandas as pd, glob
files = sorted(glob.glob("features/slices/*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=["StudyInstanceUID", "status"]) for f in files])
print(len(files), "blocos |", len(df), "cortes |", df.StudyInstanceUID.nunique(), "exames")
print(df.status.str.slice(0, 40).value_counts().head())

In [ ]:
# Copia a partição/amostra para a saída (para versionar no GitHub)
!cp -r splits results features/ 2>/dev/null; du -sh features